# Step 2: Train Student Model with Knowledge Distillation

This notebook trains a **compact student model** (~20k parameters) using knowledge distillation from the teacher model.

## Prerequisites
- Trained teacher model (`teacher_v2_robust.h5`) from Step 1
- Same ECG dataset used for teacher training

## Key Features
- Compact architecture with depthwise separable convolutions
- Knowledge distillation: learns from teacher's soft predictions
- ~12x smaller than teacher model
- Suitable for edge/mobile deployment

## How to Use This Notebook

### On Kaggle:
1. Upload your ECG dataset and teacher model
2. Update the paths in the configuration section
3. Run all cells
4. Download `student_distilled.h5` for deployment

In [ ]:
# =====================================================
# CONFIGURATION - MODIFY THESE VALUES AS NEEDED
# =====================================================

# Data paths - UPDATE THESE FOR YOUR ENVIRONMENT
# For Kaggle:
DATA_PATH = '/kaggle/input/ecg-dataset/ecg.csv'
DATA_PATH_2 = '/kaggle/input/ecg2-dataset/ecg3.csv'  # Optional (set to None if not using)
TEACHER_MODEL_PATH = '/kaggle/input/teacher-model/teacher_v2_robust.h5'  # Upload your teacher model

# For local:
# DATA_PATH = '../../ecg.csv'
# DATA_PATH_2 = '../../ecg3.csv'
# TEACHER_MODEL_PATH = '../../outputs/models/teacher_v2_robust.h5'

# Output directory
OUTPUT_DIR = '/kaggle/working'  # For Kaggle
# OUTPUT_DIR = '../../outputs/models'  # For local

# Training parameters
BATCH_SIZE = 32
EPOCHS = 200
LEARNING_RATE = 0.001
RANDOM_STATE = 42

# Distillation parameters
TEMPERATURE = 3.0  # Temperature for softening probabilities (2-4 recommended)
ALPHA = 0.7  # Weight for distillation loss (vs hard label loss)

In [ ]:
# Import libraries
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, Tuple, Optional, List
from scipy.ndimage import shift as scipy_shift

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Conv1D, GlobalAveragePooling1D, Dense, Dropout,
    BatchNormalization, Activation, Input, SeparableConv1D
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Load Teacher Model and Data

In [ ]:
# Load teacher model
print(f"Loading teacher model from: {TEACHER_MODEL_PATH}")
teacher = tf.keras.models.load_model(TEACHER_MODEL_PATH)
teacher_params = sum([np.prod(w.shape) for w in teacher.trainable_weights])
print(f"Teacher model loaded: {teacher_params:,} parameters")
teacher.summary()

In [ ]:
# Load data
print(f"\nLoading data from: {DATA_PATH}")
df1 = pd.read_csv(DATA_PATH, header=None)
print(f"Dataset 1 shape: {df1.shape}")

if DATA_PATH_2 and os.path.exists(DATA_PATH_2):
    print(f"Loading second dataset from: {DATA_PATH_2}")
    df2 = pd.read_csv(DATA_PATH_2, header=None)
    df = pd.concat([df1, df2], ignore_index=True)
    print(f"Combined dataset shape: {df.shape}")
else:
    df = df1

# Prepare data
X = df.iloc[:, :-1].values.astype(np.float32)
y = df.iloc[:, -1].values.astype(np.int32)
X = X.reshape((X.shape[0], X.shape[1], 1))
y_cat = to_categorical(y)

print(f"\nFeatures shape: {X.shape}")
print(f"Labels: Normal={np.sum(y == 0)}, Abnormal={np.sum(y == 1)}")

In [ ]:
# Split data
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, random_state=RANDOM_STATE, stratify=y_temp.argmax(axis=1)
)

print(f"Training: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")

# Class weights
y_train_classes = np.argmax(y_train, axis=1)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_classes), y=y_train_classes)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class weights: {class_weight_dict}")

## 2. Define Student Model Architecture

In [ ]:
def create_student_model(input_shape, num_classes=2):
    """
    Create compact student model with depthwise separable convolutions.
    Target: <100k parameters (actual: ~20k)
    """
    inputs = Input(shape=input_shape)
    
    # Block 1: Initial conv
    x = Conv1D(16, kernel_size=5, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    # Block 2: Depthwise separable
    x = SeparableConv1D(32, kernel_size=5, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = SeparableConv1D(32, kernel_size=3, padding='same', strides=2)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(0.1)(x)
    
    # Block 3: Depthwise separable
    x = SeparableConv1D(64, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = SeparableConv1D(64, kernel_size=3, padding='same', strides=2)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(0.15)(x)
    
    # Block 4: Final conv
    x = SeparableConv1D(96, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(0.2)(x)
    
    # Global pooling and output
    x = GlobalAveragePooling1D()(x)
    x = Dense(48, activation='relu')(x)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    return Model(inputs=inputs, outputs=outputs, name='student_model')


# Create student model
input_shape = (X_train.shape[1], 1)
num_classes = y_train.shape[1]

student = create_student_model(input_shape, num_classes)
student_params = sum([np.prod(w.shape) for w in student.trainable_weights])

print(f"\nStudent model created: {student_params:,} parameters")
print(f"Compression ratio: {teacher_params/student_params:.1f}x smaller than teacher")
student.summary()

## 3. Knowledge Distillation Training

In [ ]:
class DistillationLoss:
    """Combined distillation loss: KL divergence + Cross-entropy."""
    
    def __init__(self, temperature=3.0, alpha=0.7):
        self.temperature = temperature
        self.alpha = alpha
        self.kl_loss = tf.keras.losses.KLDivergence()
        self.ce_loss = tf.keras.losses.CategoricalCrossentropy()
    
    def __call__(self, y_true, y_pred_student, y_pred_teacher):
        # Soft targets (temperature-scaled)
        soft_teacher = tf.nn.softmax(tf.math.log(y_pred_teacher + 1e-10) / self.temperature)
        soft_student = tf.nn.softmax(tf.math.log(y_pred_student + 1e-10) / self.temperature)
        
        # KL divergence (scaled by T^2)
        kl_loss = self.kl_loss(soft_teacher, soft_student) * (self.temperature ** 2)
        
        # Hard label loss
        hard_loss = self.ce_loss(y_true, y_pred_student)
        
        return self.alpha * kl_loss + (1 - self.alpha) * hard_loss


distill_loss = DistillationLoss(temperature=TEMPERATURE, alpha=ALPHA)
print(f"Distillation loss initialized: T={TEMPERATURE}, alpha={ALPHA}")

In [ ]:
# Custom training loop for distillation
student.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

optimizer = Adam(learning_rate=LEARNING_RATE)

# Training history
history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

best_val_loss = float('inf')
patience_counter = 0
patience = 30

n_batches = int(np.ceil(len(X_train) / BATCH_SIZE))

print(f"\nStarting distillation training...")
print(f"Batches per epoch: {n_batches}")
print("="*60)

In [ ]:
# Training loop
os.makedirs(OUTPUT_DIR, exist_ok=True)

for epoch in range(EPOCHS):
    # Shuffle training data
    indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]
    
    epoch_loss = 0
    epoch_acc = 0
    
    for i in range(n_batches):
        start = i * BATCH_SIZE
        end = min(start + BATCH_SIZE, len(X_train))
        X_batch = X_train_shuffled[start:end]
        y_batch = y_train_shuffled[start:end]
        
        # Get teacher predictions
        teacher_preds = tf.stop_gradient(teacher(X_batch, training=False))
        
        # Training step
        with tf.GradientTape() as tape:
            student_preds = student(X_batch, training=True)
            loss = distill_loss(y_batch, student_preds, teacher_preds)
        
        gradients = tape.gradient(loss, student.trainable_variables)
        optimizer.apply_gradients(zip(gradients, student.trainable_variables))
        
        # Accuracy
        y_pred = tf.argmax(student_preds, axis=1)
        y_true = tf.argmax(y_batch, axis=1)
        acc = tf.reduce_mean(tf.cast(y_pred == y_true, tf.float32))
        
        epoch_loss += float(loss)
        epoch_acc += float(acc)
    
    epoch_loss /= n_batches
    epoch_acc /= n_batches
    
    # Validation
    val_results = student.evaluate(X_val, y_val, verbose=0)
    val_loss, val_acc = val_results[0], val_results[1]
    
    history['loss'].append(epoch_loss)
    history['accuracy'].append(epoch_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)
    
    # Print progress
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} - loss: {epoch_loss:.4f} - acc: {epoch_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        student.save(os.path.join(OUTPUT_DIR, 'student_distilled.h5'))
        if (epoch + 1) % 10 == 0:
            print(f"  -> Saved best model (val_loss: {val_loss:.4f})")
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print("="*60)
print("Training complete!")

In [ ]:
# Load best student model
student = tf.keras.models.load_model(os.path.join(OUTPUT_DIR, 'student_distilled.h5'))
print("Loaded best student model")

## 4. Train Baseline Model (No Distillation)

In [ ]:
# Train baseline tiny model without distillation for comparison
print("\nTraining baseline model (no distillation) for comparison...")
print("="*60)

baseline = create_student_model(input_shape, num_classes)
baseline.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

baseline_callbacks = [
    ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'baseline_tiny.h5'),
        monitor='val_loss', save_best_only=True, mode='min', verbose=0
    ),
    EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=0)
]

baseline_history = baseline.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=baseline_callbacks,
    class_weight=class_weight_dict,
    verbose=0
)

print("Baseline training complete!")

## 5. Evaluate and Compare Models

In [ ]:
def evaluate_model(model, X_test, y_test, name="Model"):
    """Evaluate model and return metrics."""
    y_true = np.argmax(y_test, axis=1)
    y_pred_proba = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    acc = np.mean(y_pred == y_true)
    f1 = f1_score(y_true, y_pred, pos_label=1)
    try:
        auc = roc_auc_score(y_true, y_pred_proba[:, 1])
    except:
        auc = 0.0
    
    # Measure inference time
    times = []
    for _ in range(100):
        start = time.perf_counter()
        model.predict(X_test[:1], verbose=0)
        times.append((time.perf_counter() - start) * 1000)
    inference_time = np.mean(times)
    
    params = sum([np.prod(w.shape) for w in model.trainable_weights])
    
    return {
        'name': name,
        'params': params,
        'accuracy': acc,
        'f1': f1,
        'auc': auc,
        'inference_ms': inference_time,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }


# Evaluate all models
print("Evaluating models...")
teacher_results = evaluate_model(teacher, X_test, y_test, "Teacher")
student_results = evaluate_model(student, X_test, y_test, "Student (Distilled)")
baseline_results = evaluate_model(baseline, X_test, y_test, "Baseline (Tiny)")

print("Evaluation complete!")

In [ ]:
# Print comparison table
print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(f"{'Metric':<25} {'Teacher':>12} {'Student':>12} {'Baseline':>12}")
print("-"*70)
print(f"{'Parameters':.<25} {teacher_results['params']:>12,} {student_results['params']:>12,} {baseline_results['params']:>12,}")
print(f"{'Accuracy':.<25} {teacher_results['accuracy']:>12.4f} {student_results['accuracy']:>12.4f} {baseline_results['accuracy']:>12.4f}")
print(f"{'AUC':.<25} {teacher_results['auc']:>12.4f} {student_results['auc']:>12.4f} {baseline_results['auc']:>12.4f}")
print(f"{'F1 (Abnormal)':.<25} {teacher_results['f1']:>12.4f} {student_results['f1']:>12.4f} {baseline_results['f1']:>12.4f}")
print(f"{'Inference Time (ms)':.<25} {teacher_results['inference_ms']:>12.2f} {student_results['inference_ms']:>12.2f} {baseline_results['inference_ms']:>12.2f}")
print("="*70)

In [ ]:
# Robustness evaluation
def evaluate_robustness(model, X_test, y_test, shifts_ms=[-40, -20, 0, 20, 40]):
    input_len = X_test.shape[1]
    fs = 360
    scale = input_len / (fs * 0.8)
    y_true = np.argmax(y_test, axis=1)
    
    results = []
    for shift_ms in shifts_ms:
        shift_samples = int(shift_ms * fs / 1000 * scale)
        X_shifted = np.zeros_like(X_test)
        for i in range(len(X_test)):
            X_shifted[i] = scipy_shift(X_test[i].squeeze(), shift_samples, mode='nearest').reshape(-1, 1)
        
        y_pred = np.argmax(model.predict(X_shifted, verbose=0), axis=1)
        results.append(np.mean(y_pred == y_true))
    
    return shifts_ms, results


print("\nEvaluating robustness...")
shifts, teacher_rob = evaluate_robustness(teacher, X_test, y_test)
_, student_rob = evaluate_robustness(student, X_test, y_test)
_, baseline_rob = evaluate_robustness(baseline, X_test, y_test)

In [ ]:
# Plot robustness comparison
plt.figure(figsize=(10, 6))
plt.plot(shifts, teacher_rob, 'b-o', linewidth=2, markersize=8, label='Teacher')
plt.plot(shifts, student_rob, 'g-s', linewidth=2, markersize=8, label='Student (Distilled)')
plt.plot(shifts, baseline_rob, 'r-^', linewidth=2, markersize=8, label='Baseline (Tiny)')
plt.xlabel('Temporal Shift (ms)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Model Robustness Comparison', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'robustness_comparison.png'), dpi=150)
plt.show()

In [ ]:
# Training history plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(history['accuracy'], label='Train')
axes[0].plot(history['val_accuracy'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Student Model Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['loss'], label='Train')
axes[1].plot(history['val_loss'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Student Model Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'student_training_history.png'), dpi=150)
plt.show()

In [ ]:
# Confusion matrices
y_true = np.argmax(y_test, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, results, title in zip(axes, 
                               [teacher_results, student_results, baseline_results],
                               ['Teacher', 'Student (Distilled)', 'Baseline (Tiny)']):
    cm = confusion_matrix(y_true, results['predictions'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Abnormal'])
    disp.plot(ax=ax, cmap='Blues')
    ax.set_title(f'{title}\nAcc: {results["accuracy"]:.3f}')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices_comparison.png'), dpi=150)
plt.show()

## 6. Save Results

In [ ]:
# Save comparison results
comparison_data = {
    'Metric': ['Parameters', 'Accuracy', 'AUC', 'F1 (Abnormal)', 'Inference Time (ms)'],
    'Teacher': [teacher_results['params'], teacher_results['accuracy'], 
                teacher_results['auc'], teacher_results['f1'], teacher_results['inference_ms']],
    'Student (Distilled)': [student_results['params'], student_results['accuracy'],
                            student_results['auc'], student_results['f1'], student_results['inference_ms']],
    'Baseline (Tiny)': [baseline_results['params'], baseline_results['accuracy'],
                        baseline_results['auc'], baseline_results['f1'], baseline_results['inference_ms']]
}

# Add robustness metrics
for i, shift in enumerate(shifts):
    comparison_data['Metric'].append(f'Acc @ {shift}ms')
    comparison_data['Teacher'].append(teacher_rob[i])
    comparison_data['Student (Distilled)'].append(student_rob[i])
    comparison_data['Baseline (Tiny)'].append(baseline_rob[i])

comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison.csv'), index=False)
print(f"Results saved to: {os.path.join(OUTPUT_DIR, 'model_comparison.csv')}")

In [ ]:
# Final summary
print("\n" + "="*60)
print("TRAINING COMPLETE - SUMMARY")
print("="*60)
print(f"\nModels saved:")
print(f"  - {os.path.join(OUTPUT_DIR, 'student_distilled.h5')} (USE THIS FOR DEPLOYMENT)")
print(f"  - {os.path.join(OUTPUT_DIR, 'baseline_tiny.h5')}")
print(f"\nPlots saved:")
print(f"  - {os.path.join(OUTPUT_DIR, 'robustness_comparison.png')}")
print(f"  - {os.path.join(OUTPUT_DIR, 'student_training_history.png')}")
print(f"  - {os.path.join(OUTPUT_DIR, 'confusion_matrices_comparison.png')}")
print(f"\nData saved:")
print(f"  - {os.path.join(OUTPUT_DIR, 'model_comparison.csv')}")
print(f"\nKey Results:")
print(f"  - Student achieves {student_results['accuracy']/teacher_results['accuracy']*100:.1f}% of teacher accuracy")
print(f"  - Student is {teacher_results['params']/student_results['params']:.1f}x smaller")
print(f"  - Student is {teacher_results['inference_ms']/student_results['inference_ms']:.1f}x faster")

## Next Steps

After training the student model:

1. **Download the model**: Download `student_distilled.h5` from `/kaggle/working/`

2. **For deployment**:
   - Use `deploy.py` to process continuous ECG recordings
   - Use `export_tflite.py` to convert to TFLite for mobile deployment

3. **For mobile/edge deployment**:
   - Convert to TFLite INT8 for smallest size
   - Integrate with your mobile app using TensorFlow Lite